# Phase 1 Task B — EfficientNet-B3 × 3 Seeds

**Model:** EfficientNet-B3
**Seeds:** 42, 123, 456
**Batch:** 32
**Output:** wandb `woodbench-taskB`

Chạy notebook này trên 1 GPU (vast.ai).

In [ ]:
from pathlib import Path
NOTEBOOK = Path("__file__").resolve()
ROOT = NOTEBOOK
for _ in range(10):
    ROOT = ROOT.parent
    if (ROOT / "pyproject.toml").exists() or (ROOT / ".git").exists(): break
import sys, os
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
print(f"[ROOT] {ROOT}")

In [ ]:
from dotenv import load_dotenv
load_dotenv(dotenv_path := ROOT / ".env")
print(f"[.env] Loaded from {dotenv_path}")

import subprocess, sys
for pkg in ["timm", "albumentations", "scikit-learn", "seaborn", "wandb", "ptflops"]:
    try: __import__(pkg.replace("-", "_"))
    except: subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

os.environ.setdefault("WANDB_API_KEY", os.getenv("WANDB_API_KEY", ""))
print(f"[wandb] API key: {'set' if os.getenv('WANDB_API_KEY') else 'MISSING'}")

In [ ]:
from src.data import get_dataloaders
DATA_ROOT = ROOT / "data" / "task-B"
tl, vl, testl = get_dataloaders(str(DATA_ROOT), batch_size=32, num_workers=4, task="taskB")
print(f"[Data] Train={len(tl.dataset)}, Val={len(vl.dataset)}, Test={len(testl.dataset)}")
print(f"[Data] Classes={len(tl.dataset.classes)}")

In [ ]:
import wandb, torch, numpy as np
from src.models import create_model
from src.train import TrainingEngine, create_optimizer, create_scheduler, create_loss_function
from src.utils import set_seed

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[Device] {DEVICE}")

In [ ]:
def train_one(backbone, seed, data_root, batch_size=32, epochs=50):
    set_seed(seed)
    output_dir = ROOT / "data" / "experiments" / "taskB" / backbone / f"seed_{seed}"
    output_dir.mkdir(parents=True, exist_ok=True)

    train_loader, val_loader, _testl_inner = get_dataloaders(
        data_root=data_root, batch_size=batch_size, num_workers=4, task="taskB"
    )
    _globals()["_testl"] = _testl_inner
    num_classes = len(train_loader.dataset.classes)
    print(
        f"[{backbone} s{seed}] train={len(train_loader.dataset)}, classes={num_classes}"
    )

    run = wandb.init(
        project="woodbench-taskB",
        name=f"{backbone}_s{seed}",
        config={
            "backbone": backbone,
            "seed": seed,
            "epochs": epochs,
            "batch_size": batch_size,
            "num_classes": num_classes,
            "lr_head": 1e-4,
            "lr_backbone": 1e-5,
            "label_smoothing": 0.1,
            "scheduler": "cosine",
            "amp": True,
        },
        dir=str(output_dir),
    )

    model = create_model(backbone, num_classes=num_classes, pretrained=True)
    optimizer = create_optimizer(model, lr=1e-4, weight_decay=1e-4, backbone_lr=1e-5)
    scheduler = create_scheduler(optimizer, epochs=epochs, scheduler_type="cosine")
    criterion = create_loss_function("label_smoothing", label_smoothing=0.1)

    engine = TrainingEngine(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        optimizer=optimizer,
        scheduler=scheduler,
        criterion=criterion,
        device=DEVICE,
        use_amp=True,
    )
    result = engine.fit(
        epochs=epochs,
        output_dir=output_dir,
        name=f"{backbone}_seed{seed}",
        save_best_only=True,
        early_stopping_patience=10,
    )
    run.finish()
    print(f"[{backbone} s{seed}] val_acc={result['best_val_acc']:.2f}%")
    return result

In [ ]:
SEEDS = [42, 123, 456]
results = {}

for seed in SEEDS:
    print(f"\n{'='*60}")
    print(f"EfficientNet-B3 | seed={seed} ({SEEDS.index(seed)+1}/{len(SEEDS)})")
    print(f"{'='*60}")
    results[seed] = train_one("efficientnet_b3", seed, str(DATA_ROOT), batch_size=32, epochs=50)

print("\n" + "="*60)
print("EfficientNet-B3 — Summary")
for seed, res in results.items():
    print(f"  seed={seed}: {res['best_val_acc']:.2f}%")
accs = [r['best_val_acc'] for r in results.values()]
print(f"  Mean: {np.mean(accs):.2f}% ± {np.std(accs):.2f}%")

In [ ]:
# ── Cell 7: Confusion Matrix + Test Metrics (all seeds) ──────────────────────
import json, matplotlib, torch, numpy as np, matplotlib.pyplot as plt, wandb
from sklearn.metrics import ConfusionMatrixDisplay, classification_report

from src.data import get_dataloaders
DATA_ROOT = ROOT / "data" / "task-B"
_ignored_train, _ignored_val, test_loader = get_dataloaders(
    str(DATA_ROOT), batch_size=32, num_workers=4, task="taskB")
class_names = test_loader.dataset.classes
num_classes = len(class_names)
print(f"[CM] {num_classes} classes")

SEEDS = [42, 123, 456]
all_preds, all_labels = {}, {}

for seed in SEEDS:
    backbone_dir = ROOT / "experiments" / "taskB" / "efficientnet_b3" / f"seed_{seed}"
    ckpt_candidates = sorted(backbone_dir.glob("*best*.pth"))
    if not ckpt_candidates:
        print(f"[CM] No checkpoint found for seed={seed}, skipping"); continue
    ckpt_path = ckpt_candidates[0]
    print(f"[CM] Loading: {ckpt_path}")

    from src.models import create_model
    model = create_model("efficientnet_b3", num_classes=num_classes, pretrained=False)
    ckpt = torch.load(ckpt_path, map_location="cpu")
    model.load_state_dict(ckpt["model_state_dict"])
    model = model.to(DEVICE).eval()

    preds_s, labels_s = [], []
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(DEVICE)
            outputs = model(images)
            preds_s.extend(outputs.argmax(1).cpu().numpy())
            labels_s.extend(labels.numpy())
    all_preds[seed] = np.array(preds_s)
    all_labels[seed] = np.array(labels_s)

    from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
    acc = accuracy_score(labels_s, preds_s)
    f1  = f1_score(labels_s, preds_s, average="macro", zero_division=0)
    prec = precision_score(labels_s, preds_s, average="macro", zero_division=0)
    rec  = recall_score(labels_s, preds_s, average="macro", zero_division=0)
    f1_per = f1_score(labels_s, preds_s, average=None, zero_division=0)
    print(f"[CM s{seed}] Acc={acc*100:.2f}%  F1={f1*100:.2f}%  Prec={prec*100:.2f}%  Rec={rec*100:.2f}%")
    for cn, fi in zip(class_names, f1_per):
        print(f"  {cn}: F1={fi*100:.2f}%")

    wandb.init(project="woodbench-taskB", name=f"efficientnet_b3_s{seed}_test",
               id=f"efficientnet_b3_s{seed}_test", resume="must", dir=str(backbone_dir))
    wandb.log({"test/accuracy": acc, "test/precision": prec, "test/recall": rec, "test/f1": f1,
               **{f"test/f1/{cn}": float(fi) for cn, fi in zip(class_names, f1_per)}})
    wandb.finish()

best_seed = max(all_preds, key=lambda s: accuracy_score(all_labels[s], all_preds[s]))
y_true, y_pred = all_labels[best_seed], all_preds[best_seed]
fig, ax = plt.subplots(figsize=(18, 14))
ConfusionMatrixDisplay.from_predictions(y_true, y_pred, display_labels=class_names,
    cmap="Blues", normalize="true", ax=ax, values_format=".2f")
ax.set_title(f"EfficientNet-B3 — Confusion Matrix (Test Set, seed={best_seed})", fontsize=14)
plt.xticks(rotation=45, ha="right"); plt.tight_layout()
cm_path = ROOT / "experiments" / "taskB" / "efficientnet_b3" / "confusion_matrix.png"
fig.savefig(cm_path, dpi=150, bbox_inches="tight")
print(f"[CM] Saved to {cm_path}")
wandb.init(project="woodbench-taskB", name="efficientnet_b3_CM",
           dir=str(ROOT / "experiments" / "taskB" / "efficientnet_b3"))
wandb.log({"confusion_matrix": wandb.Image(str(cm_path))}); wandb.finish(); plt.show()
print("\n" + "="*80)
print("Classification Report (best seed =", best_seed, ")")
print("="*80)
print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))